# Lab 2 — Exploring and Analysing Word Embeddings

*Statistical Foundations of LLMs · Session 1 · Accompanies slides 93–96*

N-gram models treat words as unrelated symbols. **Word embeddings** fix this: every word becomes a vector, and *similar words get similar vectors*. In this activity you will train your own Word2Vec model, compare it with pretrained GloVe vectors, do vector arithmetic ("king − man + woman ≈ ?"), and probe where static embeddings break down.


> ✅ **SOLUTIONS NOTEBOOK.** Every exercise stub is filled in and every challenge cell contains working reference code. Use this for self-check or as an instructor key — encourage students to attempt the exercises in the main notebook first.

## Learning Objectives

1. Train a **Word2Vec** model and load pretrained **GloVe** vectors.
2. Query similarity and solve **analogies** with vector arithmetic.
3. Visualize embedding space with **PCA** and **t-SNE**.
4. Diagnose how static embeddings handle **polysemous** words ("bank", "spring").
5. Compare Word2Vec vs. GloVe and articulate the limits of *static* embeddings — motivating contextual models (BERT).


## 0. New to Jupyter? Start Here (2 minutes)

**What is a Jupyter Notebook?** A document that mixes text and runnable Python code, organized in *cells*.

| What you need to know | How |
|---|---|
| **Run a cell** | Click it, then press **Shift + Enter** (or the ▶ button) |
| **Cell types** | **Markdown** cells = formatted text (like this one). **Code** cells = Python you can execute |
| **Order matters** | Run cells **top to bottom**. A cell may depend on variables defined above it |
| **Restart the kernel** | Menu: *Runtime → Restart runtime* (Colab) or *Kernel → Restart* (Jupyter). Then re-run cells from the top |
| **Install packages** | Run a cell starting with `%pip install ...`, then restart the kernel if asked |
| **Modify code** | Just edit any code cell and re-run it — experimenting is the whole point! |
| **Read outputs** | Results appear directly below each code cell: printed text, tables, or plots |

> 💡 **Tip:** If something behaves strangely, *Restart runtime* and run all cells from the top (*Runtime → Run all*).


## 1. Background

- **Word2Vec** (Mikolov et al., 2013) learns embeddings *predictively*: a small neural network predicts context words (Skip-Gram) or the center word (CBOW).
- **GloVe** (Pennington et al., 2014) learns embeddings by *factorizing* a global word co-occurrence matrix.

Both produce **one fixed vector per word** — that's why they're called *static* embeddings. Keep asking: *what goes wrong when one word has several meanings?*


## 2. Setup and Imports

⏳ The GloVe download (~66 MB) and Word2Vec training (~1–2 min) run while you read the next section — start this cell now.


In [ ]:
# Run once if packages are missing:
# %pip install gensim scikit-learn matplotlib numpy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gensim.downloader as api
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

np.random.seed(42)
print("Imports OK ✓")


## 3. Part A — Train Your Own Word2Vec

Instead of downloading the 1.7 GB GoogleNews vectors, we train Word2Vec **live** on `text8` (first 100 MB of Wikipedia, ~17M words). Watching a model train beats downloading one!


In [ ]:
corpus = api.load("text8")          # streams tokenized Wikipedia sentences (~31 MB download)

w2v_model = Word2Vec(
    corpus,
    vector_size=100,   # embedding dimension
    window=5,          # context window (Skip-Gram default)
    min_count=5,       # ignore very rare words
    workers=4,
    seed=42,
    epochs=3,
)
word2vec = w2v_model.wv  # the KeyedVectors (what we query)
print(f"Trained! Vocabulary: {len(word2vec):,} words, dimension: {word2vec.vector_size}")


### 3.1 Similarity queries

`most_similar` returns nearest neighbours by **cosine similarity**.


In [ ]:
print("Nearest to 'king': ", word2vec.most_similar("king", topn=5))
print()
print("sim(car, truck)  =", round(word2vec.similarity("car", "truck"), 3))
print("sim(car, banana) =", round(word2vec.similarity("car", "banana"), 3))


### 3.2 Analogies via vector arithmetic

The famous result: $\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$. In gensim: `positive=["king", "woman"], negative=["man"]`.


In [ ]:
print("king - man + woman   ≈", word2vec.most_similar(positive=["king", "woman"], negative=["man"], topn=3))
print("paris - france + italy ≈", word2vec.most_similar(positive=["paris", "italy"], negative=["france"], topn=3))
print("programmer - keyboard + brush ≈", word2vec.most_similar(positive=["programmer", "brush"], negative=["keyboard"], topn=3))


### ✏️ Exercise 3.1 — Design your own analogies

Create **three analogies from your own field** (statistics, biology, art, ...). For each, note whether the answer makes sense — and try one you expect to *fail*.


In [ ]:
# Three analogies from different domains (answers depend on the trained vectors).
def analogy(a, b, c):
    """Solve a - b + c and print the top-3 candidates."""
    try:
        return word2vec.most_similar(positive=[a, c], negative=[b], topn=3)
    except KeyError as e:
        return f"out-of-vocabulary: {e}"

print("Statistics: mean - average + median   =>", analogy("mean", "average", "median"))
print("Geography : london - england + japan  =>", analogy("london", "england", "japan"))
print("Biology   : dog - puppy + cat         =>", analogy("dog", "puppy", "cat"))
# Expect one to look weak: analogies work best for relations well represented in
# the corpus (capitals, gender) and worse for rare or fuzzy relations.

### 3.3 Visualize clusters with PCA

Embeddings live in 100 dimensions; **PCA** projects them to 2D while preserving as much variance as possible. Do the themes form clusters?


In [ ]:
THEMES = {
    "royalty": ["king", "queen", "prince", "princess", "throne"],
    "animals": ["dog", "cat", "horse", "lion", "elephant"],
    "countries": ["france", "italy", "japan", "brazil", "korea"],
    "science": ["physics", "chemistry", "biology", "mathematics", "statistics"],
}

def plot_themes(kv, themes, title):
    words = [w for ws in themes.values() for w in ws if w in kv]
    X = np.stack([kv[w] for w in words])
    Z = PCA(n_components=2, random_state=0).fit_transform(X)
    plt.figure(figsize=(9, 7))
    i = 0
    for theme, ws in themes.items():
        ws = [w for w in ws if w in kv]
        pts = Z[i:i + len(ws)]
        plt.scatter(pts[:, 0], pts[:, 1], label=theme, s=60)
        for j, w in enumerate(ws):
            plt.annotate(w, pts[j], fontsize=10, xytext=(4, 4), textcoords="offset points")
        i += len(ws)
    plt.legend(); plt.title(title); plt.axis("off"); plt.show()

plot_themes(word2vec, THEMES, "Word2Vec (self-trained on text8) — PCA projection")


### ✏️ Exercise 3.2

Add a **fifth theme of your own** to `THEMES` (e.g., foods, sports, emotions) and re-run the plot. Does your theme cluster tightly? If not, hypothesize why (frequency? ambiguity? multi-topic usage?).

> 💡 Also try replacing `PCA` with `TSNE(n_components=2, random_state=0, perplexity=5)` — t-SNE preserves *local* neighbourhoods rather than global variance.


## 4. Part B — Compare with Pretrained GloVe

Now load **GloVe** vectors trained on 6B tokens of Wikipedia + Gigaword. Same dimension (100), vastly more data — and a different learning principle (global co-occurrence counts vs. local prediction).


In [ ]:
glove = api.load("glove-wiki-gigaword-100")   # ~128 MB download, cached afterwards
print(f"GloVe loaded: {len(glove):,} words, dimension {glove.vector_size}")


In [ ]:
# Repeat the same probes on GloVe
print("king - man + woman ≈", glove.most_similar(positive=["king", "woman"], negative=["man"], topn=3))
print()
print("Nearest to 'king':", glove.most_similar("king", topn=5))


### 4.1 Head-to-head analogy test


In [ ]:
import pandas as pd

ANALOGIES = [
    (["king", "woman"], ["man"], "queen"),
    (["paris", "italy"], ["france"], "rome"),
    (["walking", "swam"], ["walked"], "swimming"),
    (["big", "smaller"], ["bigger"], "small"),
]

rows = []
for pos, neg, expected in ANALOGIES:
    for name, kv in [("Word2Vec (text8)", word2vec), ("GloVe (wiki-gigaword-100)", glove)]:
        try:
            top = kv.most_similar(positive=pos, negative=neg, topn=3)
            answer = top[0][0]
            rows.append({"analogy": f"{pos[0]} - {neg[0]} + {pos[1]}", "model": name,
                         "top-1": answer, "correct?": answer == expected,
                         "top-3": [w for w, _ in top]})
        except KeyError as e:
            rows.append({"analogy": f"{pos[0]} - {neg[0]} + {pos[1]}", "model": name,
                         "top-1": f"OOV: {e}", "correct?": False, "top-3": []})

pd.DataFrame(rows)


### 4.2 Ambiguous words: the Achilles heel of static embeddings

"bank" means *financial institution* AND *river bank* — but it gets **one single vector**. Which sense wins? Usually the more frequent one; the vector is a frequency-weighted blend of all senses.


In [ ]:
AMBIGUOUS = ["bank", "cell", "light", "spring"]

for w in AMBIGUOUS:
    print(f"--- '{w}' ---")
    print("  Word2Vec:", [t for t, _ in word2vec.most_similar(w, topn=6)])
    print("  GloVe:   ", [t for t, _ in glove.most_similar(w, topn=6)])
    print()


### ✏️ Exercise 4.1 — Sense scoring

For each ambiguous word we define "anchor" words per sense. Complete the code to measure which sense dominates each model's vector.


In [ ]:
SENSE_ANCHORS = {
    "bank":   {"financial": ["money", "loan", "credit"],   "river":  ["river", "water", "shore"]},
    "cell":   {"biology":   ["tissue", "organism", "gene"], "phone":  ["phone", "mobile", "telephone"]},
    "spring": {"season":    ["summer", "winter", "autumn"], "coil":   ["metal", "mechanical", "device"]},
    "light":  {"illumination": ["bright", "lamp", "sun"], "weight": ["heavy", "weight", "mass"]},
}


def sense_scores(kv, word):
    """Mean cosine similarity between `word` and each sense's anchor words."""
    scores = {}
    for sense, anchors in SENSE_ANCHORS[word].items():
        anchors = [a for a in anchors if a in kv]
        scores[sense] = round(float(np.mean([kv.similarity(word, a) for a in anchors])), 3)
    return scores


for w in SENSE_ANCHORS:
    print(f"{w:7} | Word2Vec: {sense_scores(word2vec, w)} | GloVe: {sense_scores(glove, w)}")
# Expected: one sense dominates each word (e.g. 'bank' leans financial). The
# minority sense is diluted because a static vector blends all senses by
# frequency — exactly what contextual models (BERT) fix.

<details><summary>💡 <b>Solution</b></summary>

```python
def sense_scores(kv, word):
    scores = {}
    for sense, anchors in SENSE_ANCHORS[word].items():
        anchors = [a for a in anchors if a in kv]
        scores[sense] = round(float(np.mean([kv.similarity(word, a) for a in anchors])), 3)
    return scores
```
**Expected observation:** one sense clearly dominates (e.g., "bank" leans financial). The minority sense is *diluted*, not represented. A contextual model like BERT instead produces a *different* vector for "bank" in each sentence.
</details>


## 5. Part C — Critical Reflection and Analysis

Write short answers (2–4 sentences each) in the Markdown cell below — this replaces the 2–3 page report from the slides and will seed our class discussion.

1. **Performance:** Which model solved more analogies? Separate two factors: *training data size* vs. *algorithm*. Which do you think mattered more here, and how could you test that?
2. **Clustering:** Did one model produce visibly better semantic groupings in the PCA plots?
3. **Ambiguity:** Summarize your sense-scoring results. What would a *contextual* embedding do differently?
4. **Limitations:** List two real applications where a static embedding would fail but contextual embeddings (BERT, next session) succeed.
5. **Personal design:** Report your best and worst custom analogy from Exercise 3.1.


### ✍️ Your answers

1. ...
2. ...
3. ...
4. ...
5. ...


## 6. 🏆 Challenge Exercises

**Challenge A — Bias probe.** Embeddings inherit biases from their corpora. Compute `glove.most_similar(positive=["doctor", "woman"], negative=["man"])` and the reverse. Try `nurse`, `engineer`, `programmer`. Document what you find — we return to this in Session 4 (ethics) and in the hallucinations & bias case study.

**Challenge B — Odd one out.** Use `kv.doesnt_match(["breakfast", "lunch", "dinner", "korea"])`. Build 5 of your own; find one where the model fails.

**Challenge C — Dimensionality.** Retrain Word2Vec with `vector_size=10` and `vector_size=300`. How does analogy accuracy change? What is the cost?


In [ ]:
# --- Challenge A: bias probe ---
print("doctor - man + woman =>", glove.most_similar(positive=["doctor", "woman"], negative=["man"], topn=3))
print("doctor - woman + man =>", glove.most_similar(positive=["doctor", "man"], negative=["woman"], topn=3))
print("nurse  - woman + man =>", glove.most_similar(positive=["nurse", "man"], negative=["woman"], topn=3))
# These often surface stereotypical associations — embeddings inherit corpus bias.
# We quantify this properly in notebook 08 (hallucinations & bias).

# --- Challenge B: odd one out ---
print("\nodd-one-out:", glove.doesnt_match(["breakfast", "lunch", "dinner", "korea"]))
print("odd-one-out:", glove.doesnt_match(["red", "green", "blue", "happy"]))

# --- Challenge C: dimensionality trade-off (conceptual) ---
# Retraining Word2Vec with vector_size=10 usually drops analogy accuracy sharply
# (too few dimensions to encode many relations); vector_size=300 improves quality
# but costs memory and compute. Diminishing returns appear well before 300.

## 7. Discussion Questions

1. Word2Vec *predicts* context; GloVe *counts* co-occurrence. Why might both arrive at similar geometry?
2. Why does vector arithmetic work at all? What property of the training objective encourages linear structure?
3. If "bank" has one vector, what happens in a downstream sentiment model for the sentence *"the fishing was great by the bank"*?
4. Our text8 Word2Vec used ~17M tokens; GloVe used 6B. GPT-4 used trillions. What does this scaling suggest about data vs. algorithms?


## Key Takeaways

- Embeddings map words to vectors where **geometry ≈ meaning**: similarity, analogy, clustering all become linear algebra.
- Word2Vec (predictive, local) and GloVe (count-based, global) produce comparable spaces; **data scale often matters more than the algorithm**.
- **Static embeddings blend all senses of a word into one vector** — the key limitation that contextual embeddings (BERT, Session 2) resolve.
- Embeddings inherit **biases** from their training corpora — a preview of Session 4.

## References

- Mikolov et al. (2013), *Efficient Estimation of Word Representations in Vector Space* — https://arxiv.org/abs/1301.3781
- Pennington et al. (2014), *GloVe: Global Vectors for Word Representation* — https://nlp.stanford.edu/projects/glove/
- Bolukbasi et al. (2016), *Man is to Computer Programmer as Woman is to Homemaker?* — https://arxiv.org/abs/1607.06520
